In [1]:
# ╔══════════════════════════════════════════════════════════════╗
# ║  CELL 1 — Requirements & Dependency Installation with UV    ║
# ╚══════════════════════════════════════════════════════════════╝

import subprocess, sys

# ── Step 1: Install UV (Rust-based ultra-fast package manager) ──────────
subprocess.run(
    ["pip", "install", "uv", "-q"],
    check=True
)
print("✅ UV installed")

# ── Step 2: Install core packages via UV ────────────────────────────────
packages = [
    "unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git",
    "trl>=0.9.4",
    "transformers>=4.45.0",
    "datasets>=2.20.0",
    "accelerate>=0.34.0",
    "peft>=0.12.0",
    "bitsandbytes>=0.43.0",
    "sentencepiece",
    "protobuf",
    "huggingface_hub",
    "pandas",
    "numpy",
    "rich",
]

subprocess.run(
    [sys.executable, "-m", "uv", "pip", "install", "--system"]
    + packages
    + ["--index-url", "https://pypi.org/simple"],
    check=True
)
print("✅ All core packages installed via UV")

# ── Step 3: Install xformers (memory-efficient attention) ────────────────
subprocess.run(
    [sys.executable, "-m", "uv", "pip", "install",
     "--system", "xformers", "--no-deps"],
    check=True
)
print("✅ xformers installed")

# ── Step 4: Verify critical imports ─────────────────────────────────────
import torch
from unsloth import FastLanguageModel
from datasets import load_dataset
from trl import SFTTrainer, GRPOTrainer

print(f"\n🔥 PyTorch      : {torch.__version__}")
print(f"🖥️  CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"🎮  GPU           : {torch.cuda.get_device_name(0)}")
    print(f"💾  VRAM          : {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")
print("\n✅ All imports verified — ready to fine-tune!")

✅ UV installed


Using Python 3.12.12 environment at: /usr
Resolved 90 packages in 2.83s
Prepared 1 package in 718ms
Uninstalled 1 package in 16ms
Installed 1 package in 3ms
 - unsloth==2026.2.1 (from git+https://github.com/unslothai/unsloth.git@37a4c826a0d6cbd97d7e741f069ab6dc012611c6)
 + unsloth==2026.2.1 (from git+https://github.com/unslothai/unsloth.git@434b38f6e1d3b97f23d465bbdbefb53c1c835720)
Using Python 3.12.12 environment at: /usr
Resolved 1 package in 2ms
Audited 1 package in 0.48ms


✅ All core packages installed via UV
✅ xformers installed
🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.


2026-02-24 14:01:50.954426: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1771941711.095654    1013 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1771941711.142987    1013 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1771941711.493376    1013 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1771941711.493412    1013 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1771941711.493415    1013 computation_placer.cc:177] computation placer alr

🦥 Unsloth Zoo will now patch everything to make training faster!

🔥 PyTorch      : 2.9.0+cu126
🖥️  CUDA available: True
🎮  GPU           : Tesla T4
💾  VRAM          : 15.6 GB

✅ All imports verified — ready to fine-tune!


In [2]:
# ╔══════════════════════════════════════════════════════════════╗
# ║  CELL 2 — Dataset Loading & Previewing                      ║
# ╚══════════════════════════════════════════════════════════════╝

from datasets import load_dataset
from rich.console import Console
from rich.table import Table as RichTable
from rich.syntax import Syntax
from rich import print as rprint

console = Console()

# ── Load ReasonMed (SFT dataset) ────────────────────────────────────────
console.rule("[bold cyan]Loading ReasonMed Dataset[/bold cyan]")
reasonmed = load_dataset(
    "lingshu-medical-mllm/ReasonMed",
    split="train"
)
print(f"✅ ReasonMed loaded  |  Rows: {len(reasonmed):,}")
print(f"   Columns: {reasonmed.column_names}")

# ── Load MedMCQA (GRPO dataset) ─────────────────────────────────────────
console.rule("[bold green]Loading MedMCQA Dataset[/bold green]")
medmcqa = load_dataset(
    "openlifescienceai/medmcqa",
    split="train"
)
print(f"✅ MedMCQA loaded    |  Rows: {len(medmcqa):,}")
print(f"   Columns: {medmcqa.column_names}")

# ── Preview ReasonMed — one sample ──────────────────────────────────────
console.rule("[bold cyan]ReasonMed — Sample Record (index 0)[/bold cyan]")
sample_rm = reasonmed[0]
for key, val in sample_rm.items():
    preview = str(val)[:250] + "..." if len(str(val)) > 250 else str(val)
    rprint(f"  [bold yellow]{key}[/bold yellow]: {preview}")

# ── Preview MedMCQA — one sample ────────────────────────────────────────
console.rule("[bold green]MedMCQA — Sample Record (index 0)[/bold green]")
sample_mq = medmcqa[0]
for key, val in sample_mq.items():
    preview = str(val)[:250] + "..." if len(str(val)) > 250 else str(val)
    rprint(f"  [bold yellow]{key}[/bold yellow]: {preview}")

# ── Dataset Stats Summary Table ──────────────────────────────────────────
console.rule("[bold magenta]Dataset Summary[/bold magenta]")
stats = RichTable(show_header=True, header_style="bold white on dark_blue")
stats.add_column("Dataset",  style="cyan",    min_width=14)
stats.add_column("Split",    style="white",   min_width=8)
stats.add_column("Rows",     justify="right", style="green", min_width=10)
stats.add_column("Columns",  justify="center",style="white", min_width=10)
stats.add_column("Pipeline Use", style="yellow", min_width=14)

stats.add_row(
    "ReasonMed", "train",
    f"{len(reasonmed):,}",
    str(len(reasonmed.column_names)),
    "SFT (Stage 1)"
)
stats.add_row(
    "MedMCQA", "train",
    f"{len(medmcqa):,}",
    str(len(medmcqa.column_names)),
    "GRPO (Stage 2)"
)
console.print(stats)

──────────────────────────────────────────── Loading ReasonMed Dataset ────────────────────────────────────────────

✅ ReasonMed loaded  |  Rows: 1,111,555
   Columns: ['instruction', 'input', 'output']


───────────────────────────────────────────── Loading MedMCQA Dataset ─────────────────────────────────────────────

✅ MedMCQA loaded    |  Rows: 182,822
   Columns: ['id', 'question', 'opa', 'opb', 'opc', 'opd', 'cop', 'choice_type', 'exp', 'subject_name', 'topic_name']


─────────────────────────────────────── ReasonMed — Sample Record (index 0) ───────────────────────────────────────

...

exp: Chronic urethral obstruction because of urinary calculi, prostatic hyperophy, tumors, normal pregnancy, 
tumors, uterine prolapse or functional disorders cause hydronephrosis which by definition is used to describe 
dilatation of renal pelvis and calcu...

subject_name: Anatomy

topic_name: Urinary tract

───────────────────────────────────────────────── Dataset Summary ─────────────────────────────────────────────────

┏━━━━━━━━━━━━━━━━┳━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━┓
┃ Dataset        ┃ Split    ┃       Rows ┃  Columns   ┃ Pipeline Use   ┃
┡━━━━━━━━━━━━━━━━╇━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━┩
│ ReasonMed      │ train    │  1,111,555 │     3      │ SFT (Stage 1)  │
│ MedMCQA        │ train    │    182,822 │     11     │ GRPO (Stage 2) │
└────────────────┴──────────┴────────────┴────────────┴────────────────┘

In [4]:
# ╔══════════════════════════════════════════════════════════════╗
# ║  CELL 3 — Dataset Preparation & SFT Structure Formatting    ║
# ╚══════════════════════════════════════════════════════════════╝

import numpy as np
from unsloth import FastLanguageModel
from unsloth.chat_templates import get_chat_template
from rich.console import Console
from rich.syntax import Syntax
from rich import print as rprint

console = Console()

# ── 3A  Load Qwen 2.5 7B with Unsloth 4-bit (fits T4 16GB) ─────────────
console.rule("[bold blue]Loading Qwen 2.5 7B Instruct via Unsloth[/bold blue]")

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name    = "unsloth/Qwen2.5-7B-Instruct-bnb-4bit",
    max_seq_length = 1024,
    dtype         = None,       # auto: bfloat16 on Ampere+, float16 on T4
    load_in_4bit  = True,       # essential for T4 VRAM budget
)

# Apply Qwen 2.5 chat template to the tokenizer
tokenizer = get_chat_template(
    tokenizer,
    chat_template = "qwen-2.5",
)
print("✅ Model + tokenizer ready with Qwen 2.5 chat template\n")

# ── 3B  Inspect ReasonMed columns before mapping ─────────────────────────
console.rule("[bold cyan]ReasonMed — Column Names[/bold cyan]")
rprint(reasonmed.column_names)
rprint("\nSample keys/values:")
for k, v in reasonmed[0].items():
    rprint(f"  [bold]{k}[/bold]: {str(v)[:120]}")

# ── 3C  System prompt ────────────────────────────────────────────────────
SYSTEM_PROMPT = (
    "You are a highly knowledgeable medical assistant. "
    "Think step-by-step before providing your final answer. "
    "Always reason carefully about medical facts and evidence."
)

# ── 3D  Formatter: ReasonMed row → Qwen chat-template string ────────────
def format_reasonmed_for_sft(example):
    """
    Maps ReasonMed fields to a Qwen 2.5 chat conversation.
    Uses .get() with multiple fallback keys to handle schema variations.
    Reasoning is wrapped in <think> tags in the assistant turn.
    """
    # Flexible field resolution
    question  = (example.get("question")         or
                 example.get("Question")         or
                 example.get("input")            or "")

    reasoning = (example.get("reasoning")        or
                 example.get("think")            or
                 example.get("chain_of_thought") or
                 example.get("rationale")        or "")

    answer    = (example.get("answer")           or
                 example.get("response")         or
                 example.get("Answer")           or
                 example.get("output")           or "")

    # Append MCQ options to question if present
    options = example.get("options") or example.get("choices") or None
    if options and isinstance(options, dict):
        opts_str = "\n".join(f"  {k}: {v}" for k, v in options.items())
        question = f"{question}\n\nOptions:\n{opts_str}"
    elif options and isinstance(options, list):
        opts_str = "\n".join(f"  {chr(65+i)}: {o}" for i, o in enumerate(options))
        question = f"{question}\n\nOptions:\n{opts_str}"

    # Build messages in Qwen chat format
    messages = [
        {"role": "system",    "content": SYSTEM_PROMPT},
        {"role": "user",      "content": question},
        {"role": "assistant", "content": f"<think>\n{reasoning}\n</think>\n\n{answer}"},
    ]

    # Apply tokenizer chat template (returns a single formatted string)
    text = tokenizer.apply_chat_template(
        messages,
        tokenize              = False,
        add_generation_prompt = False,
    )
    return {"text": text}


# ── 3E  Apply formatter to full ReasonMed dataset ────────────────────────
console.rule("[bold cyan]Formatting ReasonMed → SFT Format[/bold cyan]")

sft_dataset = reasonmed.map(
    format_reasonmed_for_sft,
    batched = False,
    desc    = "Formatting ReasonMed for SFT",
)

# Drop all original columns — SFTTrainer only needs "text"
sft_dataset = sft_dataset.remove_columns(
    [c for c in sft_dataset.column_names if c != "text"]
)

print(f"✅ SFT dataset ready  |  Rows: {len(sft_dataset):,}")
print(f"   Columns: {sft_dataset.column_names}")

# ── 3F  Display 2 formatted samples ─────────────────────────────────────
for idx in [0, 1]:
    console.rule(f"[bold yellow]Formatted SFT Sample — Index {idx}[/bold yellow]")
    syn = Syntax(sft_dataset[idx]["text"], "text", theme="monokai", word_wrap=True)
    console.print(syn)

# ── 3G  Token length statistics (sampled) ────────────────────────────────
console.rule("[bold magenta]Token Length Statistics (sample of 500)[/bold magenta]")

sample_n    = min(500, len(sft_dataset))
token_lens  = [
    len(tokenizer.encode(sft_dataset[i]["text"]))
    for i in range(sample_n)
]

print(f"  Min tokens    : {np.min(token_lens)}")
print(f"  Max tokens    : {np.max(token_lens)}")
print(f"  Mean tokens   : {np.mean(token_lens):.0f}")
print(f"  Median tokens : {np.median(token_lens):.0f}")
print(f"  95th pct      : {np.percentile(token_lens, 95):.0f}")
print(f"\n  💡 Recommended max_seq_length: {int(np.percentile(token_lens, 95) // 256 + 1) * 256}")
print("\n✅ Cell 3 complete — sft_dataset is ready for SFTTrainer!")

──────────────────────────────────── Loading Qwen 2.5 7B Instruct via Unsloth ─────────────────────────────────────

==((====))==  Unsloth 2026.2.1: Fast Qwen2 patching. Transformers: 4.57.6.
   \\   /|    Tesla T4. Num GPUs = 2. Max memory: 14.563 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.9.0+cu126. CUDA: 7.5. CUDA Toolkit: 12.6. Triton: 3.5.0
\        /    Bfloat16 = FALSE. FA [Xformers = None. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

added_tokens.json:   0%|          | 0.00/605 [00:00<?, ?B/s]

...

<|im_start|>system                                                                                                 
You are a highly knowledgeable medical assistant. Think step-by-step before providing your final answer. Always    
reason carefully about medical facts and evidence.<|im_end|>                                                       
<|im_start|>user                                                                                                   
<|im_end|>                                                                                                         
<|im_start|>assistant                                                                                              
<think>                                                                                                            
                                                                                                                   
</think>                                                                                                           
                                                                                                                   
The patient is a factory worker presenting with excessive salivation, blue lines on the gums, tremors, disturbed   
personality, insomnia, and loss of appetite. These symptoms collectively suggest a form of heavy metal poisoning.  
To determine the most likely poisoning, it is essential to meticulously analyze each symptom in the context of the 
provided options: Mercury, Lead, Arsenic, and Phosphorus.                                                          
                                                                                                                   
Excessive salivation, also known as ptyalism, is a significant symptom to consider. This symptom is notably        
associated with mercury poisoning, particularly with elemental mercury exposure, which can disrupt autonomic       
functions leading to increased salivation. Tremors, especially fine intention tremors, are another hallmark of     
mercury toxicity, reflecting its profound effects on the central nervous system. Disturbed personality and insomnia
further support the neurological impact of mercury, aligning with the erethism syndrome—a classic presentation of  
chronic mercury poisoning characterized by behavioral changes, irritability, and insomnia. Loss of appetite is a   
non-specific symptom but can be present in various toxic exposures, including mercury.                             
                                                                                                                   
The presence of blue lines on the gums, known as Burton's lines, is traditionally a classic indicator of lead      
poisoning. These lines result from the deposition of lead sulfide at the gum margins, providing a distinctive      
clinical sign. Additionally, lead poisoning often presents with peripheral neuropathy, which can manifest as       
weakness or, in some cases, tremors. However, lead toxicity is more commonly associated with symptoms such as      
abdominal pain, constipation, and anemia rather than excessive salivation and significant personality changes.     
                                                                                                                   
Arsenic poisoning typically presents with acute gastrointestinal distress, including nausea, vomiting, diarrhea,   
and abdominal pain. Chronic exposure may lead to skin changes, such as hyperpigmentation and keratosis, as well as 
an increased risk of various cancers. While neurological symptoms can occur, they are generally not as prominent as
in mercury or lead poisoning. The absence of characteristic skin lesions and the prominence of neurological        
symptoms make arsenic a less likely culprit in this scenario.                                                      
                                                        

───────────────────────────────────────── Formatted SFT Sample — Index 1 ──────────────────────────────────────────

<|im_start|>system                                                                                                 
You are a highly knowledgeable medical assistant. Think step-by-step before providing your final answer. Always    
reason carefully about medical facts and evidence.<|im_end|>                                                       
<|im_start|>user                                                                                                   
<|im_end|>                                                                                                         
<|im_start|>assistant                                                                                              
<think>                                                                                                            
                                                                                                                   
</think>                                                                                                           
                                                                                                                   
Rett's syndrome is a severe neurodevelopmental disorder that predominantly affects females and is characterized by 
a regression in cognitive and motor skills after a period of apparently normal development. It typically manifests 
in early childhood, with symptoms becoming noticeable between the ages of 6 to 18 months. The condition involves a 
range of neurological symptoms, including loss of purposeful hand movements, slowed growth, seizures, and          
difficulties with communication and social interaction. Importantly, Rett's syndrome is a genetic disorder caused  
by mutations in the MECP2 gene, which is located on the X chromosome. This gene plays a critical role in brain     
development and function by regulating the expression of other genes essential for neuronal function.              
                                                                                                                   
In analyzing the question, it's essential to recognize that Rett's syndrome is fundamentally a genetic disorder    
rather than a condition resulting from a deficiency of a specific nutrient or vitamin. The provided options—Niacin,
Biotin, Carotene, and Vitamin D—are all vitamins, each with distinct roles in bodily functions. Niacin (vitamin B3)
is crucial for energy metabolism and DNA repair, Biotin (vitamin B7) is important for carbohydrate, fat, and       
protein metabolism, Carotene is a precursor to vitamin A and is vital for vision and immune function, and Vitamin D
is essential for calcium homeostasis and bone health. None of these vitamins are directly implicated in the        
pathogenesis of Rett's syndrome.                                                                                   
                                                                                                                   
Reviewing the original chain-of-thought, there was a misinterpretation of the disease's etiology by focusing on    
vitamin deficiencies. Rett's syndrome's cause is not linked to nutritional deficiencies but rather to a specific   
genetic mutation. The MECP2 gene mutation disrupts normal brain development and function, leading to the clinical  
manifestations observed in Rett's syndrome. This genetic basis is well-documented in medical literature and is a   
cornerstone in understanding the disease's pathophysiology.                                                        
                                                                                                                   
Option-by-option analysis reveals the following:                                                                   
                                                                                                                   
1. **Niacin**: Deficiency in Niacin leads to pellagra, w

───────────────────────────────────── Token Length Statistics (sample of 500) ─────────────────────────────────────

  Min tokens    : 502
  Max tokens    : 2835
  Mean tokens   : 1507
  Median tokens : 1483
  95th pct      : 2165

  💡 Recommended max_seq_length: 2304

✅ Cell 3 complete — sft_dataset is ready for SFTTrainer!


In [5]:
# ╔══════════════════════════════════════════════════════════════╗
# ║  CELL 4 — LoRA Config + Memory Check                        ║
# ╚══════════════════════════════════════════════════════════════╝
import torch
from unsloth import FastLanguageModel
from rich.table import Table as RichTable
from rich import print as rprint

console.rule("[bold magenta]Applying LoRA to Model[/bold magenta]")

model = FastLanguageModel.get_peft_model(
    model,
    r                   = 16,
    target_modules      = ["q_proj","k_proj","v_proj","o_proj",
                           "gate_proj","up_proj","down_proj"],
    lora_alpha          = 16,
    lora_dropout        = 0,        # 0 is optimised in Unsloth
    bias                = "none",
    use_gradient_checkpointing = "unsloth",   # long-context saving
    random_state        = 42,
    use_rslora          = False,
    loftq_config        = None,
)

# ── Trainable params ─────────────────────────────────────────────────────
trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
total     = sum(p.numel() for p in model.parameters())

p_table = RichTable(show_header=True, header_style="bold white on dark_blue")
p_table.add_column("Type",         style="cyan",   min_width=20)
p_table.add_column("Parameters",   style="green",  justify="right", min_width=16)
p_table.add_column("% of Total",   style="yellow", justify="right", min_width=12)
p_table.add_row("Trainable (LoRA)", f"{trainable:,}",       f"{100*trainable/total:.3f}%")
p_table.add_row("Frozen",           f"{total-trainable:,}", f"{100*(total-trainable)/total:.3f}%")
p_table.add_row("Total",            f"{total:,}",            "100.000%")
console.rule("[bold magenta]Parameter Summary[/bold magenta]")
console.print(p_table)

# ── GPU memory ───────────────────────────────────────────────────────────
alloc    = torch.cuda.memory_allocated()  / 1024**3
reserved = torch.cuda.memory_reserved()   / 1024**3
total_v  = torch.cuda.get_device_properties(0).total_memory / 1024**3
free     = total_v - reserved

m_table = RichTable(show_header=True, header_style="bold white on dark_blue")
m_table.add_column("Metric",     style="cyan",  min_width=20)
m_table.add_column("GB",         style="green", justify="right", min_width=10)
m_table.add_row("Total VRAM",    f"{total_v:.2f}")
m_table.add_row("Allocated",     f"{alloc:.2f}")
m_table.add_row("Reserved",      f"{reserved:.2f}")
m_table.add_row("Free (est.)",   f"{free:.2f}")
console.rule("[bold magenta]GPU Memory After LoRA[/bold magenta]")
console.print(m_table)

status = "[bold green]✅ Safe to train[/bold green]" if free > 2.5 \
    else "[bold red]⚠️  Tight — reduce max_seq_length or batch[/bold red]"
rprint(f"\n  Status: {status}")

───────────────────────────────────────────── Applying LoRA to Model ──────────────────────────────────────────────

Unsloth 2026.2.1 patched 28 layers with 28 QKV layers, 28 O layers and 28 MLP layers.


──────────────────────────────────────────────── Parameter Summary ────────────────────────────────────────────────

┏━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━┓
┃ Type                 ┃       Parameters ┃   % of Total ┃
┡━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━┩
│ Trainable (LoRA)     │       40,370,176 │       0.919% │
│ Frozen               │    4,352,972,288 │      99.081% │
│ Total                │    4,393,342,464 │     100.000% │
└──────────────────────┴──────────────────┴──────────────┘

────────────────────────────────────────────── GPU Memory After LoRA ──────────────────────────────────────────────

┏━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┓
┃ Metric               ┃         GB ┃
┡━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━┩
│ Total VRAM           │      14.56 │
│ Allocated            │       5.37 │
│ Reserved             │       5.39 │
│ Free (est.)          │       9.17 │
└──────────────────────┴────────────┘

Status: ✅ Safe to train

In [6]:
# ╔══════════════════════════════════════════════════════════════╗
# ║  CELL 5 — SFT Training Arguments                            ║
# ╚══════════════════════════════════════════════════════════════╝
from trl import SFTTrainer, SFTConfig
from rich.table import Table as RichTable

console.rule("[bold cyan]SFT Training Arguments[/bold cyan]")

# Use the recommended max_seq_length from Cell 3 token stats
# (95th percentile rounded up to nearest 256)
MAX_SEQ = 2048   # adjust down to 1024 if you see OOM

training_args = SFTConfig(
    output_dir                  = "./qwen2.5-7b-reasonmed-sft",
    num_train_epochs            = 1,
    per_device_train_batch_size = 2,
    gradient_accumulation_steps = 4,           # effective batch = 8
    warmup_ratio                = 0.05,
    learning_rate               = 2e-4,
    lr_scheduler_type           = "cosine",
    fp16                        = not torch.cuda.is_bf16_supported(),
    bf16                        = torch.cuda.is_bf16_supported(),
    optim                       = "adamw_8bit",
    weight_decay                = 0.01,
    max_seq_length              = MAX_SEQ,
    dataset_text_field          = "text",
    packing                     = True,        # Unsloth packing = faster
    logging_steps               = 25,
    save_strategy               = "epoch",
    save_total_limit            = 1,
    seed                        = 42,
    report_to                   = "none",
)

# ── Config summary ───────────────────────────────────────────────────────
cfg = RichTable(show_header=True, header_style="bold white on dark_blue")
cfg.add_column("Hyperparameter",   style="cyan",  min_width=28)
cfg.add_column("Value",            style="green", min_width=16)
for k, v in [
    ("Epochs",                      training_args.num_train_epochs),
    ("Batch / device",              training_args.per_device_train_batch_size),
    ("Grad accumulation",           training_args.gradient_accumulation_steps),
    ("Effective batch size",        training_args.per_device_train_batch_size *
                                    training_args.gradient_accumulation_steps),
    ("Learning rate",               training_args.learning_rate),
    ("LR scheduler",                training_args.lr_scheduler_type),
    ("Max seq length",              training_args.max_seq_length),
    ("Packing",                     training_args.packing),
    ("Precision",                   "bf16" if training_args.bf16 else "fp16"),
    ("Optimizer",                   training_args.optim),
    ("Output dir",                  training_args.output_dir),
]:
    cfg.add_row(str(k), str(v))
console.print(cfg)
print("✅ Training arguments ready.")

───────────────────────────────────────────── SFT Training Arguments ──────────────────────────────────────────────

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━┓
┃ Hyperparameter               ┃ Value                      ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━┩
│ Epochs                       │ 1                          │
│ Batch / device               │ 2                          │
│ Grad accumulation            │ 4                          │
│ Effective batch size         │ 8                          │
│ Learning rate                │ 0.0002                     │
│ LR scheduler                 │ SchedulerType.COSINE       │
│ Max seq length               │ 2048                       │
│ Packing                      │ True                       │
│ Precision                    │ fp16                       │
│ Optimizer                    │ OptimizerNames.ADAMW_8BIT  │
│ Output dir                   │ ./qwen2.5-7b-reasonmed-sft │
└──────────────────────────────┴────────────────────────────┘

✅ Training arguments ready.


In [ ]:
# ╔══════════════════════════════════════════════════════════════╗
# ║  CELL 6 — SFT Training (5K rows)                            ║
# ╚══════════════════════════════════════════════════════════════╝
import time, gc, torch
from trl import SFTTrainer
from rich.table import Table as RichTable

gc.collect(); torch.cuda.empty_cache()

# ── Slice to 5K rows ─────────────────────────────────────────────────────
SFT_TRAIN_ROWS = 5000
sft_5k = sft_dataset.shuffle(seed=42).select(range(SFT_TRAIN_ROWS))

console.rule("[bold green]Starting SFT Training[/bold green]")
rprint(f"  [bold]Full dataset rows :[/bold] {len(sft_dataset):,}")
rprint(f"  [bold]Training on       :[/bold] {len(sft_5k):,} rows (5K subset)")
rprint(f"  [bold]Max seq len       :[/bold] {training_args.max_seq_length}")
rprint(f"  [bold]Epochs            :[/bold] {training_args.num_train_epochs}\n")

trainer = SFTTrainer(
    model         = model,
    tokenizer     = tokenizer,
    train_dataset = sft_5k,          # ← 5K subset
    args          = training_args,
)

start        = time.time()
train_result = trainer.train()
elapsed      = time.time() - start

# ── Results ──────────────────────────────────────────────────────────────
console.rule("[bold green]Training Complete ✅[/bold green]")
m = train_result.metrics

res = RichTable(show_header=True, header_style="bold white on dark_blue")
res.add_column("Metric",   style="cyan",  min_width=24)
res.add_column("Value",    style="green", min_width=16)
res.add_row("Training rows",     f"{len(sft_5k):,}")
res.add_row("Train loss",        f"{m.get('train_loss', 0):.4f}")
res.add_row("Runtime",           f"{elapsed/60:.1f} min")
res.add_row("Samples / sec",     f"{m.get('train_samples_per_second', 0):.2f}")
res.add_row("Steps / sec",       f"{m.get('train_steps_per_second',  0):.3f}")
res.add_row("Total steps",       str(m.get('train_global_step', '—')))
res.add_row("Peak GPU mem (GB)", f"{torch.cuda.max_memory_allocated()/1024**3:.2f}")
console.print(res)

# ── Save adapter ─────────────────────────────────────────────────────────
SAVE_PATH = "./qwen2.5-7b-reasonmed-sft-adapter"
trainer.save_model(SAVE_PATH)
tokenizer.save_pretrained(SAVE_PATH)
rprint(f"\n  [bold green]💾 LoRA adapter saved →[/bold green] {SAVE_PATH}")
rprint("  [bold green]✅ Stage 1 SFT complete — ready for GRPO (Stage 2)[/bold green]")

────────────────────────────────────────────── Starting SFT Training ──────────────────────────────────────────────

Full dataset rows : 1,111,555

Training on       : 5,000 rows (5K subset)

Max seq len       : 2048

Epochs            : 1

...

Unsloth: Packing train dataset (num_proc=8):   0%|          | 0/5000 [00:00<?, ? examples/s]

🦥 Unsloth: Packing enabled - training is >2x faster and uses less VRAM!


==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 2,822 | Num Epochs = 1 | Total steps = 353
O^O/ \_/ \    Batch size per device = 2 | Gradient accumulation steps = 4
\        /    Data Parallel GPUs = 1 | Total batch size (2 x 4 x 1) = 8
 "-____-"     Trainable parameters = 40,370,176 of 7,655,986,688 (0.53% trained)


Unsloth: Will smartly offload gradients to save VRAM!


Step,Training Loss


In [ ]:
import os
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"

import torch
print(f"Free VRAM  : {(torch.cuda.get_device_properties(0).total_memory - torch.cuda.memory_allocated()) / 1e9:.2f} GB")
print(f"Total VRAM : {torch.cuda.get_device_properties(0).total_memory / 1e9:.2f} GB")